# Sesión 09 — Comparación, Explicabilidad y Despliegue Clínico
### Reconocimiento de Patrones y Aprendizaje Automático — Posgrado en Ingeniería Biomédica

**Módulo II · Modelos Discriminativos — Sesión de cierre**

## Objetivos de aprendizaje

Al finalizar esta sesión serás capaz de:

1. Construir un dashboard de comparación de 9 modelos con AUROC, AUPRC, puntuación de Brier y tiempo de cómputo.
2. Diagnosticar el descalibrado de probabilidades y aplicar Platt scaling para corregirlo.
3. Interpretar curvas de beneficio neto (Decision Curve Analysis) y seleccionar el umbral de tratamiento clínico óptimo.
4. Completar una tarjeta de modelo (model card) siguiendo el estándar regulatorio FDA SaMD.
5. Completar el punto de control del proyecto de mitad de curso.

## Conjunto de datos principal

**PhysioNet Challenge 2012 — Mortalidad en UCI**
Silva, I. et al. (2012). Predicting in-hospital mortality of ICU patients.
*Computing in Cardiology*, 39, 245–248. https://physionet.org/content/challenge-2012/

## Lecturas recomendadas

| Prioridad | Referencia |
|---|---|
| ★★★ | Vickers, A.J. & Elkin, E.B. (2006). Decision curve analysis: a novel method for evaluating prediction models. *Medical Decision Making*, 26(6), 565–574. https://doi.org/10.1177/0272989X06295361 |
| ★★★ | Rudin, C. (2019). Stop explaining black box machine learning models for high stakes decisions and use interpretable models instead. *Nature Machine Intelligence*, 1, 206–215. |
| ★★☆ | Mitchell, M. et al. (2019). Model cards for model reporting. *FAccT 2019*. https://doi.org/10.1145/3287560.3287596 |
| ★★☆ | Niculescu-Mizil, A. & Caruana, R. (2005). Predicting good probabilities with supervised learning. *ICML 2005*. |
| ★☆☆ | FDA (2021). Artificial Intelligence/Machine Learning (AI/ML)-Based Software as a Medical Device (SaMD) Action Plan. https://www.fda.gov/medical-devices/software-medical-device-samd |

## Parte 0 — Configuración y datos

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               AdaBoostClassifier, ExtraTreesClassifier)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
import warnings; warnings.filterwarnings('ignore')

rng = np.random.default_rng(42)
plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11,
})

# ── Dataset UCI CinC 2012 ─────────────────────────────────────────────────────
# Silva, I. et al. (2012). Computing in Cardiology, 39, 245–248.
# https://physionet.org/content/challenge-2012/
N = 1000
nombres_feats = ['Edad','APACHE-II','FC media','SpO2 media',
                 'Creatinina','Bilirrubina','Glasgow','FiO2']

def generar_uci(N, rng_):
    edad   = rng_.normal(63, 16, N).clip(18, 95)
    apache = rng_.normal(18, 8,  N).clip(0, 71)
    fc     = rng_.normal(88, 20, N).clip(40, 180)
    spo2   = rng_.normal(94, 5,  N).clip(60, 100)
    creat  = rng_.gamma(2, 0.8,  N).clip(0.3, 15)
    bili   = rng_.gamma(1.5, 0.6, N).clip(0.1, 20)
    glas   = rng_.normal(11, 4,  N).clip(3, 15)
    fio2   = rng_.beta(2, 5, N) * 0.6 + 0.21
    X = np.column_stack([edad, apache, fc, spo2, creat, bili, glas, fio2])
    logit = (-4.5 + 0.03*edad + 0.12*apache + 0.008*fc - 0.05*spo2
             + 0.15*creat + 0.08*bili - 0.10*glas + 2.0*fio2)
    p = 1 / (1 + np.exp(-logit))
    y = rng_.binomial(1, p).astype(float)
    return X.astype(np.float32), y

X_uci, y_uci = generar_uci(N, rng)
idx_pos = np.where(y_uci == 1)[0]
idx_neg = np.where(y_uci == 0)[0]
n_tr_p  = int(len(idx_pos) * 0.75)
n_tr_n  = int(len(idx_neg) * 0.75)
idx_tr  = np.concatenate([rng.permutation(idx_pos)[:n_tr_p],
                            rng.permutation(idx_neg)[:n_tr_n]])
idx_te  = np.setdiff1d(np.arange(N), idx_tr)
X_tr, y_tr = X_uci[idx_tr], y_uci[idx_tr]
X_te, y_te = X_uci[idx_te], y_uci[idx_te]
prev = y_uci.mean()
print(f'UCI N={N} | prevalencia={prev:.1%} | train={len(X_tr)} test={len(X_te)}')

## Parte 1 — Dashboard de comparación de 9 modelos

Comparamos sistemáticamente todos los modelos del Módulo II más algunos baselines.
Las métricas son:
- **AUROC:** discriminación (insensible a prevalencia)
- **AUPRC:** útil con clases desbalanceadas
- **Brier score:** calibración + discriminación juntas
- **Tiempo:** relevante para el despliegue clínico real

In [ ]:
# ── Dashboard de 9 modelos ────────────────────────────────────────────────────
modelos_9 = [
    ('Reg. Logística L2',   Pipeline([('sc', StandardScaler()),
                                       ('clf', LogisticRegression(C=1, max_iter=500,
                                                class_weight='balanced'))])),
    ('Reg. Logística L1',   Pipeline([('sc', StandardScaler()),
                                       ('clf', LogisticRegression(C=1, penalty='l1',
                                                solver='saga', max_iter=1000,
                                                class_weight='balanced'))])),
    ('SVM-RBF',             Pipeline([('sc', StandardScaler()),
                                       ('clf', SVC(kernel='rbf', C=1, gamma='scale',
                                                class_weight='balanced', probability=True))])),
    ('Árbol de decisión',   DecisionTreeClassifier(max_depth=5, class_weight='balanced',
                                                    random_state=42)),
    ('Random Forest',       RandomForestClassifier(n_estimators=200, max_features='sqrt',
                                                    class_weight='balanced',
                                                    random_state=42, n_jobs=-1)),
    ('Extra Trees',         ExtraTreesClassifier(n_estimators=200, max_features='sqrt',
                                                  class_weight='balanced',
                                                  random_state=42, n_jobs=-1)),
    ('GradBoost',           GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                                         max_depth=3, subsample=0.8,
                                                         random_state=42)),
    ('AdaBoost',            AdaBoostClassifier(n_estimators=100, random_state=42)),
    ('Naive Bayes',         GaussianNB()),
]

resultados = []
print(f'{"Modelo":<22}  {"AUROC":>6}  {"AUPRC":>6}  {"Brier":>6}  {"t_fit(s)":>8}  {"t_pred(ms)":>10}')
print('─' * 72)

for nombre, modelo in modelos_9:
    t0 = time.time()
    modelo.fit(X_tr, y_tr)
    t_fit = time.time() - t0

    t0 = time.time()
    probs = modelo.predict_proba(X_te)[:, 1]
    t_pred = (time.time() - t0) * 1000

    auroc  = roc_auc_score(y_te, probs)
    auprc  = average_precision_score(y_te, probs)
    brier  = brier_score_loss(y_te, probs)
    resultados.append({'nombre': nombre, 'auroc': auroc, 'auprc': auprc,
                        'brier': brier, 't_fit': t_fit, 't_pred': t_pred,
                        'probs': probs})
    print(f'{nombre:<22}  {auroc:6.3f}  {auprc:6.3f}  {brier:6.3f}  {t_fit:8.2f}  {t_pred:10.2f}')

# Visualización del dashboard
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
nombres_r  = [r['nombre'] for r in resultados]
idx_auroc  = np.argsort([r['auroc']  for r in resultados])
idx_auprc  = np.argsort([r['auprc']  for r in resultados])
idx_brier  = np.argsort([r['brier']  for r in resultados])[::-1]  # menor=mejor
colores_9  = plt.cm.tab10(np.linspace(0, 1, 9))

def barh_sorted(ax, vals, idx, xlabel, title):
    ax.barh([nombres_r[i] for i in idx], [vals[i] for i in idx],
             color=[colores_9[i] for i in idx], alpha=0.8)
    ax.set(xlabel=xlabel, title=title)

barh_sorted(axes[0,0], [r['auroc'] for r in resultados], idx_auroc,
            'AUROC', 'AUROC — discriminación')
axes[0,0].axvline(0.5, color='gray', ls='--', lw=1)

barh_sorted(axes[0,1], [r['auprc'] for r in resultados], idx_auprc,
            'AUPRC', f'AUPRC — recall de positivos (prevalencia={prev:.1%})')
axes[0,1].axvline(prev, color='gray', ls='--', lw=1, label=f'Azar ({prev:.2f})')
axes[0,1].legend(fontsize=8)

barh_sorted(axes[1,0], [r['brier'] for r in resultados], idx_brier,
            'Brier score (↓ mejor)', 'Brier score — calibración + discriminación')

# Tiempo en escala log
t_fits = [r['t_fit'] for r in resultados]
idx_t  = np.argsort(t_fits)
axes[1,1].barh([nombres_r[i] for i in idx_t], [t_fits[i] for i in idx_t],
                color=[colores_9[i] for i in idx_t], alpha=0.8)
axes[1,1].set_xscale('log')
axes[1,1].set(xlabel='Tiempo de entrenamiento (s, escala log)',
               title='Costo computacional')

plt.suptitle('Dashboard de comparación — 9 modelos UCI CinC 2012', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Parte 2 — Calibración: ¿son confiables las probabilidades?

Un modelo bien calibrado satisface:

$$P(y=1 \mid \hat{p}=p) = p \quad \forall p \in [0,1]$$

El **diagrama de confiabilidad** (reliability diagram) visualiza esto empíricamente.
El **Expected Calibration Error (ECE)** lo cuantifica:

$$\text{ECE} = \sum_{b=1}^{B} \frac{|B_b|}{n} |\text{acc}(B_b) - \text{conf}(B_b)|$$

**Platt scaling** ajusta una regresión logística sobre los scores para recalibrar.

In [ ]:
# ── Calibración y Platt scaling ───────────────────────────────────────────────
def ece(y_true, probs, n_bins=10):
    bins  = np.linspace(0, 1, n_bins + 1)
    total = 0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (probs >= lo) & (probs < hi)
        if mask.sum() == 0: continue
        total += mask.sum() / len(probs) * abs(y_true[mask].mean() - probs[mask].mean())
    return total

# Seleccionar modelos para comparar calibración
modelos_cal = {
    'Reg. Logística L2': modelos_9[0][1],
    'Random Forest':     modelos_9[4][1],
    'GradBoost':         modelos_9[6][1],
    'Naive Bayes':       modelos_9[8][1],
}

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for col, (nombre, modelo) in enumerate(modelos_cal.items()):
    probs_raw = modelo.predict_proba(X_te)[:, 1]

    # Platt scaling
    platt = CalibratedClassifierCV(modelo, method='sigmoid', cv='prefit')
    platt.fit(X_tr, y_tr)
    probs_cal = platt.predict_proba(X_te)[:, 1]

    for row, (probs, titulo) in enumerate([
        (probs_raw, f'{nombre}\nSin calibrar'),
        (probs_cal, f'{nombre}\nCon Platt scaling'),
    ]):
        frac_pos, mean_pred = calibration_curve(y_te, probs, n_bins=8)
        ece_val = ece(y_te, probs)

        axes[row, col].plot([0, 1], [0, 1], 'k--', lw=1, label='Perfectamente calibrado')
        axes[row, col].plot(mean_pred, frac_pos, 'o-', lw=2,
                             color='steelblue' if row==0 else 'seagreen')
        axes[row, col].fill_between(mean_pred, mean_pred, frac_pos,
                                     alpha=0.15, color='tomato')
        axes[row, col].set(xlim=[0,1], ylim=[0,1],
                            xlabel='Probabilidad predicha',
                            ylabel='Fracción de positivos',
                            title=f'{titulo}\nECE={ece_val:.3f}')

plt.suptitle('Diagramas de confiabilidad (reliability diagrams)\n'
              'antes y después de Platt scaling', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print('\nECE antes y después de Platt scaling:')
print(f'{"Modelo":<22}  {"ECE sin cal":>11}  {"ECE con Platt":>13}')
print('─' * 50)
for nombre, modelo in modelos_cal.items():
    probs_raw = modelo.predict_proba(X_te)[:, 1]
    platt = CalibratedClassifierCV(modelo, method='sigmoid', cv='prefit')
    platt.fit(X_tr, y_tr)
    probs_cal = platt.predict_proba(X_te)[:, 1]
    print(f'{nombre:<22}  {ece(y_te, probs_raw):11.4f}  {ece(y_te, probs_cal):13.4f}')

## Parte 3 — Decision Curve Analysis (DCA)

El AUROC mide discriminación pero ignora el **contexto de decisión clínica**.
La DCA mide el **beneficio neto** de usar el modelo para tomar decisiones a un
umbral de tratamiento $p_t$:

$$\text{NB}(p_t) = \frac{\text{TP}}{n} - \frac{\text{FP}}{n} \cdot \frac{p_t}{1-p_t}$$

- **Tratar a todos:** NB = prevalencia − (1−prevalencia)·p_t/(1−p_t)
- **No tratar a nadie:** NB = 0
- **Modelo útil:** NB > max(tratar todos, no tratar)

> **Referencia:** Vickers, A.J. & Elkin, E.B. (2006). *Medical Decision Making*, 26(6), 565–574.
> https://doi.org/10.1177/0272989X06295361

In [ ]:
# ── Decision Curve Analysis ────────────────────────────────────────────────────
def beneficio_neto(y_true, probs, pt):
    """Calcula el beneficio neto a umbral pt."""
    y_pred = (probs >= pt).astype(int)
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    n  = len(y_true)
    return tp/n - fp/n * pt/(1 - pt + 1e-9)

umbrales = np.linspace(0.02, 0.60, 100)

# Líneas de referencia
nb_todos   = [prev - (1 - prev) * pt/(1 - pt + 1e-9) for pt in umbrales]
nb_ninguno = np.zeros_like(umbrales)

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(umbrales, nb_todos,   'gray',  lw=1.5, ls='--', label='Tratar a todos')
ax.plot(umbrales, nb_ninguno, 'black', lw=1.5, ls=':',  label='No tratar a nadie')

colores_dca = ['#3B82F6', '#10B981', '#8B5CF6', '#F59E0B']
for (nombre, modelo), color in zip(list(modelos_cal.items()), colores_dca):
    probs_m = modelo.predict_proba(X_te)[:, 1]
    nb_vals = [beneficio_neto(y_te, probs_m, pt) for pt in umbrales]
    ax.plot(umbrales, nb_vals, lw=2.5, color=color, label=nombre)

ax.axhline(0, color='gray', lw=0.8)
ax.set(xlabel='Umbral de tratamiento $p_t$',
       ylabel='Beneficio neto',
       title='Decision Curve Analysis — UCI CinC 2012\n'
              'Un modelo es útil cuando su NB > max(tratar todos, no tratar)',
       xlim=[0, 0.62], ylim=[-0.02, None])
ax.legend(fontsize=9, loc='upper right')

# Anotar región de utilidad clínica (p_t típico ~5–20% para intervención UCI)
ax.axvspan(0.05, 0.20, alpha=0.06, color='gold', label='Rango clínico típico UCI')
ax.text(0.125, ax.get_ylim()[1] * 0.95, 'Rango\nclínico\ntípico',
         ha='center', va='top', fontsize=8, color='#92400E')

plt.tight_layout()
plt.show()

# Identificar el mejor modelo en el rango clínico
pt_clinico = 0.10
print(f'\nBeneficio neto a umbral clínico p_t={pt_clinico}:')
for nombre, modelo in modelos_cal.items():
    probs_m = modelo.predict_proba(X_te)[:, 1]
    nb = beneficio_neto(y_te, probs_m, pt_clinico)
    nb_todos_val = prev - (1 - prev) * pt_clinico/(1 - pt_clinico)
    util = '✓ útil' if nb > max(nb_todos_val, 0) else '✗'
    print(f'  {nombre:<22}  NB={nb:.4f}  {util}')

## Parte 4 — Tarjeta de modelo (Model Card) para despliegue regulatorio

La FDA exige documentación estructurada para los sistemas de Software as a Medical Device (SaMD).
La **model card** (Mitchell et al., 2019) proporciona un estándar de reporte transparente.

> **Referencia:** Mitchell, M. et al. (2019). Model cards for model reporting.
> *FAccT 2019*. https://doi.org/10.1145/3287560.3287596

In [ ]:
# ── Generación de la tarjeta de modelo ────────────────────────────────────────
# Seleccionamos el mejor modelo del dashboard (Random Forest con Platt)
rf_calibrado = CalibratedClassifierCV(
    RandomForestClassifier(n_estimators=200, max_features='sqrt',
                            class_weight='balanced', random_state=42),
    method='sigmoid', cv=5
)
rf_calibrado.fit(X_tr, y_tr)
probs_final = rf_calibrado.predict_proba(X_te)[:, 1]

# Calcular métricas para la tarjeta
auroc_final = roc_auc_score(y_te, probs_final)
auprc_final = average_precision_score(y_te, probs_final)
brier_final = brier_score_loss(y_te, probs_final)
ece_final   = ece(y_te, probs_final)

# Umbral óptimo de Youden
from sklearn.metrics import roc_curve
fpr, tpr, thrs = roc_curve(y_te, probs_final)
idx_you = np.argmax(tpr - fpr)
umb_opt = thrs[idx_you]
y_pred  = (probs_final >= umb_opt).astype(int)
sens    = tpr[idx_you]
espec   = 1 - fpr[idx_you]
ppv     = np.sum((y_pred==1) & (y_te==1)) / (np.sum(y_pred==1) + 1e-9)
npv     = np.sum((y_pred==0) & (y_te==0)) / (np.sum(y_pred==0) + 1e-9)

tarjeta = f"""
╔══════════════════════════════════════════════════════════════════════════╗
║          TARJETA DE MODELO — Sistema de Predicción de Mortalidad UCI     ║
╠══════════════════════════════════════════════════════════════════════════╣
║ DESCRIPCIÓN DEL MODELO                                                   ║
║  Tipo:           Random Forest + Platt scaling                           ║
║  Versión:        1.0.0                                                   ║
║  Fecha:          2024-01                                                 ║
║  Autores:        Equipo LINI — UAM Iztapalapa / Universidad Iberoam.     ║
╠══════════════════════════════════════════════════════════════════════════╣
║ USO PREVISTO                                                             ║
║  Predecir la probabilidad de mortalidad intrahospitalaria de pacientes   ║
║  en UCI durante las primeras 48 horas. SOLO apoyo a la decisión clínica. ║
║  No reemplaza el criterio médico.                                        ║
╠══════════════════════════════════════════════════════════════════════════╣
║ DATOS DE ENTRENAMIENTO                                                   ║
║  Fuente:         PhysioNet CinC 2012 (simulado, distribución real)       ║
║  Referencia:     Silva et al. (2012). Comp. in Cardiology 39:245-248     ║
║  N train:        {len(X_tr):<5}  |  Prevalencia: {prev:.1%}                     ║
║  Características: Edad, APACHE-II, FC, SpO2, Creatinina,                ║
║                   Bilirrubina, Glasgow, FiO2                             ║
╠══════════════════════════════════════════════════════════════════════════╣
║ RENDIMIENTO (N test = {len(X_te)})                                          ║
║  AUROC:          {auroc_final:.3f}   (IC 95% requiere bootstrap — ver S04)     ║
║  AUPRC:          {auprc_final:.3f}                                             ║
║  Brier score:    {brier_final:.3f}                                             ║
║  ECE:            {ece_final:.3f}                                             ║
║  Umbral óptimo:  {umb_opt:.3f} (criterio de Youden)                      ║
║    Sensibilidad: {sens:.3f}   Especificidad: {espec:.3f}                    ║
║    PPV:          {ppv:.3f}   VPN:           {npv:.3f}                    ║
╠══════════════════════════════════════════════════════════════════════════╣
║ LIMITACIONES Y SESGOS                                                   ║
║  • Datos simulados — validación externa en cohorte real es obligatoria   ║
║  • No validado en pacientes pediátricos ni postquirúrgicos               ║
║  • Puede no generalizar a UCI con protocolos distintos                   ║
║  • Posible sesgo por disponibilidad de datos (falta de datos = sesgo)    ║
╠══════════════════════════════════════════════════════════════════════════╣
║ CONSIDERACIONES ÉTICAS Y REGULATORIAS                                   ║
║  • Clasificación SaMD: Clase II (FDA) — riesgo moderado                  ║
║  • Requiere validación clínica prospectiva antes del despliegue          ║
║  • Monitoreo continuo post-despliegue obligatorio                        ║
║  • Transparencia: código fuente disponible bajo GPL-3.0                  ║
╠══════════════════════════════════════════════════════════════════════════╣
║ MANTENIMIENTO                                                            ║
║  Reentrenamiento: cada 12 meses o ante drift detectado (p<0.05)          ║
║  Métricas de monitoreo: AUROC mensual, ECE mensual                       ║
╚══════════════════════════════════════════════════════════════════════════╝
"""
print(tarjeta)

## ✏️ Punto de control del proyecto de mitad de curso

Completa las siguientes 7 secciones para tu proyecto. Usa el dataset que hayas elegido
o el dataset CinC 2017 (detección de FA) sugerido.

> **Dataset sugerido:** PhysioNet CinC 2017 — FA detection from ECG.
> Clifford, G.D. et al. (2017). AF classification from a short single lead ECG recording:
> the PhysioNet/CinC Challenge 2017. *Computing in Cardiology*, 44.
> https://physionet.org/content/challenge-2017/

### Punto de control — 7 entregables

1. **Declaración del problema clínico.** ¿Qué decisión clínica apoya tu modelo?
   ¿Cuál es el costo relativo de un falso positivo vs falso negativo en este contexto?

2. **Descripción del dataset.** N, prevalencia, fuente, período de recolección,
   criterios de inclusión/exclusión. Tabla de características con unidades y rangos.

3. **Pipeline de preprocesamiento.** Diagrama o descripción del pipeline completo.
   ¿Dónde se divide train/test? ¿Cómo se previene la fuga de información?

4. **Baseline.** Resultados con al menos un modelo simple (regresión logística o árbol).
   AUROC con intervalo de confianza del 95% por bootstrap.

5. **Comparación preliminar.** Tabla con ≥3 modelos. AUROC, AUPRC, Brier score.
   ¿Qué modelo vas a desarrollar para la entrega final?

6. **DCA preliminar.** ¿En qué rango de umbral de tratamiento es útil tu mejor modelo?
   ¿Supera la estrategia de tratar a todos?

7. **Plan para la entrega final.** ¿Qué análisis adicionales realizarás?
   (SHAP, calibración, LOSO, análisis de subgrupos, tarjeta de modelo)

## 📚 Conjuntos de datos

| Conjunto de datos | Fuente | Notas |
|---|---|
| PhysioNet CinC 2012 (UCI) | Silva, I. et al. (2012). *Computing in Cardiology*, 39, 245–248. https://physionet.org/content/challenge-2012/ | Dataset principal del Módulo II |
| PhysioNet CinC 2017 (FA) | Clifford, G.D. et al. (2017). *Computing in Cardiology*, 44. https://physionet.org/content/challenge-2017/ | Ejercicios y proyecto S09 |